# History dynamics replay
Magnitude, changing task sensitivity, gradient ages, and adaptive scaling.

Keep this folder separate from the source experiment. It waits for the original worker to become idle and freezes the completed trajectories available then. For all 12 trajectories, launch after the original run finishes.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'run_replay.py').exists():
    ROOT = ROOT / 'history_dynamics_replay'
if not (ROOT / 'run_replay.py').exists():
    raise FileNotFoundError('Open the notebook inside the extracted history_dynamics_replay folder.')
sys.path.insert(0, str(ROOT))
for name in [p.stem for p in ROOT.glob('*.py')]:
    mod = sys.modules.get(name)
    if mod is not None and Path(getattr(mod, '__file__', '')).parent.resolve() != ROOT.resolve():
        del sys.modules[name]
import study as replay
SOURCE = Path('/home/ubuntu/9/runs/natural_forgetting_components_v1')
OUTPUT = ROOT / 'runs' / 'history_dynamics_replay_v1'
saved = OUTPUT / 'settings.json'
SETTINGS = json.loads(saved.read_text()) if saved.exists() else replay.defaults(SOURCE, OUTPUT)
SETTINGS['python'] = sys.executable
SETTINGS['hours'] = 23.0
# Optional BEFORE the first launch: SETTINGS['test_population'] = 128
replay.validate(SETTINGS)
print('Original results (read only):', SETTINGS['source'])
print('Replay output:', SETTINGS['output'])


## Launch / resume
If the original worker is active, status will say waiting. Do not restart either experiment to resume.

In [ ]:
replay.launch(SETTINGS)


## Status — refresh manually

In [ ]:
print(json.dumps(replay.status(SETTINGS), indent=2))


## Pause — only run if wanted

In [ ]:
# print(replay.stop(SETTINGS))


## Results
Compute a summary from a consistent snapshot; original training remains untouched.

In [ ]:
if (Path(SETTINGS['output']) / 'results.sqlite').exists():
    replay.show_results({'output': replay.analyze(SETTINGS)})
else:
    print('No replay measurements yet.')


## Export

In [ ]:
from IPython.display import display, FileLink
p = Path(replay.export(SETTINGS))
print(p)
display(FileLink(str(p.relative_to(Path.cwd()))))


## Last log messages — use if status failed

In [ ]:
p = Path(SETTINGS['output']) / 'worker.log'
print('\n'.join(p.read_text(errors='replace').splitlines()[-40:]) if p.exists() else 'No log yet.')
